# Model Experiments - Epstein Accountability Index

Attribution: Scaffolded with AI assistance (Claude, Anthropic)

This notebook experiments with different model architectures and hyperparameters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import joblib

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

## Load Data and Models

In [ ]:
# Load features and consequences
features_df = pd.read_csv('../data/processed/features.csv')
consequences_df = pd.read_csv('../data/processed/consequences.csv')

# Merge
df = features_df.merge(consequences_df[['name', 'consequence_tier']], on='name', how='inner')

print(f"Dataset size: {len(df)}")
print(f"Class distribution:\n{df['consequence_tier'].value_counts()}")

In [ ]:
# Prepare features and labels
feature_cols = [
    'mention_count', 'total_mentions', 'mean_context_sentiment',
    'cooccurrence_score', 'doc_type_diversity',
    'name_in_subject_line', 'severity_score'
]

X = df[feature_cols].fillna(0)
y = df['consequence_tier']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set: {len(X_train)}")
print(f"Test set: {len(X_test)}")

## Load Trained Models

In [ ]:
# Load XGBoost model
xgb_model = joblib.load('../models/xgboost_model.pkl')
print("Loaded XGBoost model")

## Model Evaluation

In [ ]:
# Evaluate on test set
y_pred = xgb_model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['None', 'Soft', 'Hard']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['None', 'Soft', 'Hard'],
            yticklabels=['None', 'Soft', 'Hard'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - XGBoost')
plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
# Plot feature importances
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.xlabel('Importance')
plt.title('XGBoost Feature Importances')
plt.tight_layout()
plt.show()

## Power Tier Analysis

In [ ]:
# Load experiment results
experiment_df = pd.read_csv('../data/outputs/experiment_results.csv')
print("Power Tier Experiment Results:")
experiment_df

In [ ]:
# Visualize correlations by power tier
plt.figure(figsize=(10, 6))
colors = ['green' if c > 0.5 else 'orange' if c > 0.3 else 'red' 
          for c in experiment_df['correlation']]
plt.bar(experiment_df['power_tier'], experiment_df['correlation'], color=colors)
plt.xlabel('Power Tier')
plt.ylabel('Correlation (Severity → Consequence)')
plt.title('Does Power Protect? Correlation by Power Tier')
plt.axhline(y=0, color='black', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## Error Analysis

In [ ]:
# Analyze misclassifications
test_df = X_test.copy()
test_df['actual'] = y_test.values
test_df['predicted'] = y_pred
test_df['name'] = df.loc[X_test.index, 'name'].values

# Find misclassifications
misclassified = test_df[test_df['actual'] != test_df['predicted']]

print(f"Misclassified cases: {len(misclassified)}")
print("\nExamples:")
misclassified[['name', 'actual', 'predicted', 'severity_score']].head(10)